In [ ]:
# Setup

%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

from comments import prompts
from comments.llm_labeling import LLMLabeler
from comments.static_labeling import label_unfinished_comments, label_code_snippets_comments
from shared import DifferencesEvaluator, load_dataset, save_dataset
from shared.deduplicate import JsonDeduplicator
from shared.llm_connector import Model

logging.basicConfig(level=logging.INFO)
deduplicated_file_path = None

In [ ]:
# Deduplication

input_path = "/media/martin/Big data1/datasets/pa165-git/output.json"
deduplicator = JsonDeduplicator()
file_path = Path(input_path)

deduplicated_file_path = deduplicator.deduplicate_dataset_file(file_path)

In [ ]:
# Labeling unfinished comments - LLM

UNFINISHED_COMMENT_ATTR = "unfinished_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

response_mapping = {
    "yes": 1,
    "no": 0
}

labeler = LLMLabeler(prompts.TODO_COMMENTS_INIT_PROMPT, prompts.RUN_PROMPT_1, UNFINISHED_COMMENT_ATTR, response_mapping, Model.LLAMA_3_1_8b)
await labeler.label_dataset(deduplicated_file_path)


In [ ]:
# Labeling unfinished comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_unfinished_comments(deduplicated_file_path)


In [ ]:
# Labeling unfinished comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="unfinished_comment_label", to_compare_keys=["unfinished_comment_llm", "unfinished_comment_static"], no_conflict_key="unfinished_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Labeling commented code comments - LLM

CODE_COMMENT_ATTR = "code_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

response_mapping = {
    "yes": 1,
    "no": 0
}

labeler = LLMLabeler(prompts.CODE_COMMENTS_INIT_PROMPT, prompts.RUN_PROMPT_1, CODE_COMMENT_ATTR, response_mapping, Model.CODE_LLAMA_13b_INSTRUCT)
await labeler.label_dataset(deduplicated_file_path)

In [ ]:
# Labeling commented code comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_code_snippets_comments(deduplicated_file_path)